Cell 1 — Install

In [ ]:
%pip install httpx selectolax pydantic tenacity aiolimiter pandas scikit-learn matplotlib


Cell 2 — Imports

In [2]:
import asyncio
import random
import re
import time
from urllib.parse import urljoin

import httpx
from aiolimiter import AsyncLimiter
from pydantic import BaseModel, Field
from selectolax.parser import HTMLParser
from tenacity import (
    retry,
    retry_if_exception,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)

Cell 3 — Configuration
├── BACKEND_URL
├── API_PREFIX
└── PRODUCT_ENDPOINT

In [3]:
# =========================
# Configuration
# =========================

BACKEND_URL = "http://localhost:8001"
API_PREFIX = "/api/v1"

PRODUCT_ENDPOINT = f"{BACKEND_URL}{API_PREFIX}/product"

Cell 4 — Product models

In [4]:
from typing import Any


class ProductImage(BaseModel):
    url: str
    alt: str = ""
    is_primary: bool = False
    sort_order: int = 0


class RawProduct(BaseModel):
    source: str
    source_url: str

    external_id: str | None = None

    name: str
    brand: str | None = None

    sku: str | None = None
    barcode: str | None = None
    mpn: str | None = None
    gtin: str | None = None

    price: float | None = None
    compare_at_price: float | None = None
    currency: str = "BDT"

    availability: str | None = None

    description: str = ""
    short_description: str = ""

    images: list[ProductImage] = Field(
        default_factory=list
    )

    specifications: dict[str, Any] = Field(
        default_factory=dict
    )

    attributes: dict[str, Any] = Field(
        default_factory=dict
    )

Cell 5 — HTTP client
Rate limiting + randomized delay + 429/Retry-After handling

In [ ]:
class EcommerceHttpClient:

    def __init__(
        self,
        requests_per_minute=20,
        min_delay=2,
        max_delay=4,
        max_retries=3,
    ):
        self.limiter = AsyncLimiter(
            requests_per_minute,
            60,
        )

        self.min_delay = min_delay
        self.max_delay = max_delay
        self.last_request = 0
        self.max_retries = max_retries

        self.client = httpx.AsyncClient(
            timeout=30,
            follow_redirects=True,
            headers={
                "User-Agent": "EcommerceImporter/1.0 (product data importer)"
            },
        )

    async def get(self, url):

        for attempt in range(1, self.max_retries + 1):

            response = await self._get_once(url)

            if response.status_code != 429:
                break

            if attempt == self.max_retries:
                break

            retry_after = response.headers.get("Retry-After")

            try:
                wait = float(retry_after) if retry_after else 60
            except ValueError:
                wait = 60

            print(
                f"429 received — waiting {wait}s "
                f"before retry {attempt + 1}/{self.max_retries}"
            )

            await asyncio.sleep(wait)

        response.raise_for_status()

        return response

    async def _get_once(self, url):

        async with self.limiter:

            elapsed = (
                time.monotonic()
                - self.last_request
            )

            delay = random.uniform(
                self.min_delay,
                self.max_delay,
            )

            if elapsed < delay:
                await asyncio.sleep(
                    delay - elapsed
                )

            response = await self.client.get(url)

            self.last_request = time.monotonic()

            return response

    async def post(self, url, json=None):

        async with self.limiter:

            elapsed = (
                time.monotonic()
                - self.last_request
            )

            delay = random.uniform(
                self.min_delay,
                self.max_delay,
            )

            if elapsed < delay:
                await asyncio.sleep(
                    delay - elapsed
                )

            response = await self.client.post(
                url,
                json=json,
            )

            self.last_request = time.monotonic()

            return response
    async def close(self):
        await self.client.aclose()

Cell 6 — Star Tech scraper

In [6]:
class StarTechScraper:

    BASE_URL = "https://www.startech.com.bd"

    def __init__(self, client):
        self.client = client

    async def scrape_category(self, url):

        response = await self.client.get(url)

        tree = HTMLParser(response.text)

        products = []

        for item in tree.css(".p-item"):

            product = self.parse_product_card(item)

            if product:
                products.append(product)

        return products

    def parse_product_card(self, item):

        name_node = item.css_first(".p-item-name")

        if not name_node:
            return None

        link = name_node.css_first("a")

        if not link:
            return None

        name = name_node.text(strip=True)

        href = link.attributes.get("href")

        if not href:
            return None

        url = urljoin(
            self.BASE_URL,
            href,
        )

        # --- price ---
        price = None

        price_node = item.css_first(
            "meta[itemprop='price']"
        )

        if price_node:

            value = price_node.attributes.get(
                "content"
            )

            if value:
                try:
                    price = float(value)
                except ValueError:
                    pass

        # --- availability ---
        availability = None

        availability_node = item.css_first(
            "meta[itemprop='availability']"
        )

        if availability_node:

            availability = availability_node.attributes.get(
                "content"
            )

            if availability:
                availability = availability.split("/")[-1]

        # --- image ---
        images = []

        image_node = item.css_first(
            "img[itemprop='image']"
        )

        if image_node:

            image_url = (
                image_node.attributes.get("src")
                or image_node.attributes.get("data-src")
            )

            if image_url:

                images.append(
                    ProductImage(
                        url=urljoin(
                            self.BASE_URL,
                            image_url,
                        ),
                        alt=name,
                        is_primary=True,
                    )
                )

        # --- specifications from the short-description list ---
        specifications = {}

        for li in item.css(".short-description li"):

            text = li.text(strip=True)

            if ":" not in text:
                continue

            key, value = text.split(":", 1)

            specifications[
                key.strip()
            ] = value.strip()

        # --- external id from the add-to-cart button ---
        external_id = None

        onclick_node = item.css_first(
            "[onclick*='cart.add']"
        )

        if onclick_node:

            onclick = onclick_node.attributes.get(
                "onclick",
                "",
            )

            match = re.search(
                r"cart\.add\(['\"]([^'\"]+)",
                onclick,
            )

            if match:
                external_id = match.group(1)

        return RawProduct(
            source="startech",
            source_url=url,
            external_id=external_id,
            name=name,
            price=price,
            currency="BDT",
            availability=availability,
            images=images,
            specifications=specifications,
        )

Cell 7 — Backend sender
Tenacity: 3 attempts, exponential backoff (1–10s), reraises the last error

In [ ]:
def is_retryable_error(exc):
    """Only retry transient failures: 5xx, 429, network errors.
    4xx (validation 400, duplicate 409, ...) will not fix themselves."""

    if isinstance(exc, httpx.HTTPStatusError):

        status = (
            exc.response.status_code
            if exc.response is not None
            else 0
        )

        return status >= 500 or status == 429

    return isinstance(exc, httpx.HTTPError)


@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    retry=retry_if_exception(is_retryable_error),
    reraise=True,
)
async def send_product_to_backend(product, http_client):

    payload = {
        "name": product.name,
        "productSlug": re.sub(
            r"[^a-z0-9]+",
            "-",
            product.name.lower(),
        ).strip("-"),
        "price": product.price,
        "currency": product.currency,
        "shortDescription": product.short_description,
        "description": product.description,
        "sku": product.sku if product.sku is not None else "",
        "barcode": product.barcode if product.barcode is not None else "",
        "mpn": product.mpn if product.mpn is not None else "",
        "gtin": product.gtin if product.gtin is not None else "",
        "images": [
            {
                "url": image.url,
                "alt": image.alt,
                "isPrimary": image.is_primary,
                "sortOrder": image.sort_order,
            }
            for image in product.images
        ],
        "specifications": product.specifications,
        "attributes": {
            **product.attributes,
            **(
                {"availability": product.availability}
                if product.availability
                else {}
            ),
        },
        "tags": [product.source],
        "sources": [
            {
                "name": product.source,
                "externalId": product.external_id,
                "url": product.source_url,
            }
        ],
        "status": "draft",
    }

    response = await http_client.post(
        PRODUCT_ENDPOINT,
        json=payload,
    )

    response.raise_for_status()

    return response.json()


Cell 8 — Test — scrape one category page

In [9]:
http_client = EcommerceHttpClient()

scraper = StarTechScraper(http_client)

products = await scraper.scrape_category(
    "https://www.startech.com.bd"
)

print(f"Scraped {len(products)} products")

if products:
    print(products[0].model_dump_json(indent=2))

Scraped 20 products
{
  "source": "startech",
  "source_url": "https://www.startech.com.bd/amd-ryzen-3-3200g-budget-desktop-pc",
  "external_id": null,
  "name": "AMD Ryzen 3 3200G Budget Desktop PC With Monitor",
  "brand": null,
  "sku": null,
  "barcode": null,
  "mpn": null,
  "gtin": null,
  "price": null,
  "compare_at_price": null,
  "currency": "BDT",
  "availability": null,
  "description": "",
  "short_description": "",
  "images": [],
  "specifications": {},
  "attributes": {}
}


Cell 9 — Test — scrape multiple pages

In [10]:
async def scrape_category_pages(
    scraper,
    base_url,
    max_pages=5,
):
    all_products = []

    for page in range(1, max_pages + 1):

        url = (
            base_url
            if page == 1
            else f"{base_url}?page={page}"
        )

        print(f"Scraping page {page}: {url}")

        page_products = await scraper.scrape_category(url)

        if not page_products:
            break

        all_products.extend(page_products)

    return all_products


products = await scrape_category_pages(
    scraper,
    "https://www.startech.com.bd/air-conditioner",
    max_pages=3,
)

print(f"Total products: {len(products)}")

Scraping page 1: https://www.startech.com.bd/air-conditioner
Scraping page 2: https://www.startech.com.bd/air-conditioner?page=2
Scraping page 3: https://www.startech.com.bd/air-conditioner?page=3
Total products: 60


Cell 10 — Send every scraped product to the backend

In [ ]:
async def product_exists(http_client, product_slug):
    """True if the backend already has this slug (404 = not found)."""

    try:

        response = await http_client.get(
            f"{PRODUCT_ENDPOINT}/slug/{product_slug}"
        )

        return response.status_code == 200

    except httpx.HTTPStatusError as exc:

        if exc.response is not None and exc.response.status_code == 404:
            return False

        raise


async def send_all_products_to_backend(products, http_client):
    """POST every scraped product. Returns (sent, skipped, failed) lists."""

    sent = []
    skipped = []
    failed = []

    for i, product in enumerate(products, start=1):

        product_slug = re.sub(
            r"[^a-z0-9]+",
            "-",
            product.name.lower(),
        ).strip("-")

        label = f"[{i}/{len(products)}]"

        try:

            if await product_exists(http_client, product_slug):
                skipped.append(product.name)
                print(f"{label} SKIPPED (already in backend): {product.name}")
                continue

            await send_product_to_backend(
                product,
                http_client,
            )

            sent.append(product.name)

            print(f"{label} OK: {product.name}")

        except httpx.HTTPStatusError as exc:

            status = (
                exc.response.status_code
                if exc.response is not None
                else None
            )

            if status == 409:
                # Duplicate key — inserted by a concurrent run
                skipped.append(product.name)
                print(f"{label} SKIPPED (duplicate): {product.name}")

            else:
                failed.append((product.name, repr(exc)))
                print(f"{label} FAILED: {product.name} — {exc!r}")

        except httpx.HTTPError as exc:

            failed.append((product.name, repr(exc)))

            print(f"{label} FAILED: {product.name} — {exc!r}")

    return sent, skipped, failed


In [ ]:
sent, skipped, failed = await send_all_products_to_backend(
    products,
    http_client,
)

print()
print(f"Sent: {len(sent)}, Skipped: {len(skipped)}, Failed: {len(failed)}")

if failed:
    print("Failed products:")

    for name, err in failed:
        print(f"  - {name}: {err}")

# Done — release the underlying connection pool
await http_client.close()


# Part 2 — Data Mining & Machine Learning

Part 1 collected the raw data (scraping → backend).
This part covers the core data-mining workflow on that catalog:

```
Raw data  →  Cleaning  →  EDA  →  Feature engineering
          →  Preprocessing (scaling / encoding)
          →  Modeling (regression, classification, clustering)
          →  Evaluation
```

The cells load data from the backend API (`GET /api/v1/product`).
If the backend is not running, a realistic **synthetic fallback dataset**
in exactly the same JSON shape — with deliberately messy rows — is
generated, so every step below still runs.

In [ ]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    r2_score,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

## 11 — Load data into pandas

Data comes from the same backend API Part 1 wrote to.
Whatever the source, the next cell flattens the backend Product JSON
(`product.model.js`: nested `attributes`, `specifications`, `rating`, ...)
into one tidy analysis row per product.

In [ ]:
async def fetch_all_products(http_client):

    response = await http_client.get(
        f"{BACKEND_URL}{API_PREFIX}/product"
    )

    response.raise_for_status()

    body = response.json()

    return body["data"] if isinstance(body, dict) else body


try:
    products_json = await fetch_all_products(http_client)
    DATA_SOURCE = "backend API"
except Exception as exc:
    print(f"Backend not reachable ({exc!r}) — using synthetic fallback data")
    products_json = None
    DATA_SOURCE = "synthetic fallback"

print(f"Data source: {DATA_SOURCE}")

In [ ]:
# Synthetic fallback: ~180 Star Tech-like AC products in the EXACT shape the
# backend API returns, with realistic dirt injected (missing prices,
# scrape-error outliers, duplicate rows, missing brand/availability).
if DATA_SOURCE == "synthetic fallback":

    rng = np.random.default_rng(RANDOM_STATE)

    BRANDS = [
        "Gree", "Haier", "Midea", "TCL", "Hisense",
        "Xiaomi", "Beko", "SINGER", "Smart", "Walton",
    ]
    BRAND_PREMIUM = {
        "Gree": 1.05, "Haier": 1.08, "Midea": 0.95, "TCL": 0.92,
        "Hisense": 0.90, "Xiaomi": 0.98, "Beko": 1.10,
        "SINGER": 0.85, "Smart": 0.80, "Walton": 0.78,
    }
    MODEL_CODES = ["DX", "Pro", "Elite", "Max", "Neo", "Prime", "Ultra", "Eco"]

    rows = []

    for i in range(180):
        brand = BRANDS[rng.integers(0, len(BRANDS))]
        capacity = rng.choice([1.0, 1.5, 2.0], p=[0.4, 0.4, 0.2])
        # larger units are more often inverter — a learnable real-world pattern
        inverter = rng.random() < {1.0: 0.35, 1.5: 0.55, 2.0: 0.75}[capacity]
        wifi = bool(rng.random() < 0.25 and inverter)
        model_code = MODEL_CODES[rng.integers(0, len(MODEL_CODES))]

        base = 28000 + capacity * 18000 + (12000 if inverter else 0)
        price = round(base * BRAND_PREMIUM[brand] * rng.uniform(0.88, 1.12), -2)

        name = (
            f"{brand} {capacity:g} Ton {model_code} "
            f"{'Inverter' if inverter else 'Non-Inverter'} AC"
            f"{' WiFi' if wifi else ''}"
        )

        rows.append(
            {
                "name": name,
                "productSlug": re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-"),
                "brand": brand if rng.random() > 0.05 else None,
                "price": price,
                "currency": "BDT",
                "specifications": {
                    "Cooling Capacity": f"{int(capacity * 12000):,} BTU/Hr",
                    "Refrigerant": str(rng.choice(["R-410A", "R-32"])),
                },
                "attributes": (
                    {
                        "availability": str(
                            rng.choice(["InStock", "Out of Stock"], p=[0.85, 0.15])
                        )
                    }
                    if rng.random() > 0.10
                    else {}
                ),
                "rating": {
                    "average": round(float(rng.uniform(3.4, 5.0)), 1),
                    "count": int(rng.integers(0, 60)),
                },
                "status": "draft",
                "tags": ["startech"],
            }
        )

    # --- inject the dirt real scrapes produce ---
    products_json = rows
    products_json[3]["price"] = None            # missing target
    products_json[17]["price"] = None
    products_json[5]["price"] = 850000          # scrape-error outliers
    products_json[11]["price"] = 640000
    products_json += [                          # duplicates across pages
        dict(products_json[7]),
        dict(products_json[20]),
        dict(products_json[33]),
    ]

print(f"Raw records: {len(products_json)}")

In [ ]:
def extract_capacity(name, specifications):
    """Capacity in tons from the product name ('1.5 Ton'),
    falling back to the BTU specification (12000 BTU ~ 1 ton)."""

    match = re.search(r"(\d(?:\.\d)?)\s*-?\s*Ton", str(name), re.I)

    if match:
        return float(match.group(1))

    btu = re.search(
        r"(\d{2,3})[,.]?000\s*BTU",
        str(specifications.get("Cooling Capacity", "")),
        re.I,
    )

    if btu:
        return round(int(btu.group(1)) / 12, 2)

    return np.nan


def product_records_to_frame(records):
    """Flatten backend Product JSON into one tidy row per product."""

    rows = []

    for rec in records:

        attributes = rec.get("attributes") or {}
        specifications = rec.get("specifications") or {}
        rating = rec.get("rating") or {}
        name = rec.get("name", "")
        name_lower = name.lower()

        rows.append(
            {
                "name": name,
                "productSlug": rec.get("productSlug"),
                "price": rec.get("price"),
                "brand": rec.get("brand"),
                "availability": attributes.get("availability"),
                "rating_average": rating.get("average") or 0.0,
                "rating_count": rating.get("count") or 0,
                # --- engineered features from the raw schema ---
                "capacity_tons": extract_capacity(name, specifications),
                "is_inverter": (
                    "inverter" in name_lower
                    and "non-inverter" not in name_lower
                ),
                "has_wifi": ("wifi" in name_lower) or ("wi-fi" in name_lower),
            }
        )

    return pd.DataFrame(rows)


df_raw = product_records_to_frame(products_json)

print(f"Raw shape: {df_raw.shape}")
df_raw.head()

## 12 — Data cleaning

Real scraped data is messy. Steps demonstrated here:

| Problem | Technique |
|---|---|
| Duplicate products (same item on several pages) | `drop_duplicates` on `productSlug` |
| Missing target values | drop rows without `price` |
| Scrape-error price outliers | IQR rule → cap (winsorize), not delete |
| Missing brand | regex recovery from the product name |
| Inconsistent labels (`TCL` vs `Tcl`) | canonical brand mapping |
| Missing / inconsistent availability | normalize, fill `Unknown` |

A **cleaning report** is printed after each step.

In [ ]:
df = df_raw.copy()
report = []

# 1. duplicates
before = len(df)
df = df.drop_duplicates(subset="productSlug").reset_index(drop=True)
report.append(("Duplicate slugs removed", before - len(df)))

# 2. rows without a price (the regression target)
before = len(df)
df = df[df["price"].notna()].reset_index(drop=True)
report.append(("Rows without price dropped", before - len(df)))

# 3. price outliers — IQR rule, capped instead of dropped
q1, q3 = df["price"].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = int(((df["price"] < lo) | (df["price"] > hi)).sum())
df["price"] = df["price"].clip(lo, hi)
report.append(("Price outliers capped (winsorized)", outliers))

# 4. missing brand — recover from the product name
KNOWN_BRANDS = [
    "gree", "haier", "midea", "tcl", "hisense", "xiaomi",
    "beko", "singer", "smart", "walton", "daikin",
]

def extract_brand(name):
    for token in re.findall(r"[A-Za-z]+", str(name)):
        if token.lower() in KNOWN_BRANDS:
            return token.lower()
    return "unknown"

before = int(df["brand"].isna().sum())
df["brand"] = df["brand"].fillna(df["name"].map(extract_brand))
recovered = before - int(df["brand"].isna().sum())
report.append(("Missing brands recovered from name", recovered))

# 5. canonical brand labels — 'TCL', 'Tcl' and 'tcl' are one brand
CANONICAL_BRANDS = {
    "gree": "Gree", "haier": "Haier", "midea": "Midea", "tcl": "TCL",
    "hisense": "Hisense", "xiaomi": "Xiaomi", "beko": "Beko",
    "singer": "SINGER", "smart": "Smart", "walton": "Walton",
    "daikin": "Daikin", "unknown": "Unknown",
}

before_labels = df["brand"].nunique()
df["brand"] = df["brand"].map(
    lambda b: CANONICAL_BRANDS.get(str(b).lower().strip(), str(b).strip())
)
report.append(
    (
        f"Brand labels canonicalized "
        f"({before_labels} raw labels -> {df['brand'].nunique()})",
        before_labels - df["brand"].nunique(),
    )
)

# 6. availability — canonical labels, missing stays visible as 'Unknown'
CANONICAL_AVAILABILITY = {
    "instock": "InStock",
    "outofstock": "OutOfStock",
    "out of stock": "OutOfStock",
    " preorder": "PreOrder",
}

def normalize_availability(value):
    key = str(value).strip().lower()
    if key in ("", "none", "nan"):
        return "Unknown"
    return CANONICAL_AVAILABILITY.get(key, str(value).strip())

before_labels = df["availability"].nunique()
df["availability"] = df["availability"].map(normalize_availability)
report.append(
    (
        f"Availability canonicalized "
        f"({before_labels} raw labels -> {df['availability'].nunique()})",
        int((df["availability"] == "Unknown").sum()),
    )
)

print("Cleaning report:")
for label, n in report:
    print(f"  - {label}: {n}")
print(f"\nClean shape: {df.shape}")
df[["name", "brand", "price", "capacity_tons", "is_inverter"]].head()

## 13 — Exploratory Data Analysis (EDA)

Distributions, category frequencies and price relationships
are visualized before any modeling.

In [ ]:
print(df[["price", "capacity_tons", "rating_average", "rating_count"]].describe().round(2))
print()
print("Brand frequency:")
print(df["brand"].value_counts())
print()
print("Availability:")
print(df["availability"].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("EDA — scraped product catalog", fontsize=14)

axes[0, 0].hist(df["price"], bins=25, color="#4c72b0", edgecolor="white")
axes[0, 0].set_title("Price distribution (BDT)")
axes[0, 0].set_xlabel("Price")

axes[0, 1].hist(df["capacity_tons"], bins=9, color="#55a868", edgecolor="white")
axes[0, 1].set_title("Capacity distribution (tons)")
axes[0, 1].set_xlabel("Capacity")

brand_counts = df["brand"].value_counts()
axes[1, 0].bar(brand_counts.index, brand_counts.values, color="#c44e52")
axes[1, 0].set_title("Products per brand")
axes[1, 0].tick_params(axis="x", rotation=45)

groups = [g["price"].values for _, g in df.groupby("is_inverter")]
axes[1, 1].boxplot(groups, tick_labels=["Non-Inverter", "Inverter"])
axes[1, 1].set_title("Price by inverter type")
axes[1, 1].set_ylabel("Price (BDT)")

plt.tight_layout()
plt.show()

## 14 — Feature engineering & preprocessing

Features were engineered in the flatten step:
`capacity_tons` (regex from the name, BTU fallback), `is_inverter`,
`has_wifi`, rating aggregates.

Preprocessing in a leak-free `Pipeline`:

- **Numeric** (`capacity_tons`, `rating_average`, `rating_count`)
  → median-impute missing → standardize
- **Categorical** (`brand`, `availability`) → one-hot encode (unseen categories ignored)

A single `ColumnTransformer` guarantees identical preprocessing at
train and predict time.

In [ ]:
NUMERIC_FEATURES = ["capacity_tons", "rating_average", "rating_count"]
CATEGORICAL_FEATURES = ["brand", "availability"]
TARGET = "price"

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            NUMERIC_FEATURES,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES,
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 15 — Regression — predict product price

Two models compared with **5-fold cross-validation** (R²):
`LinearRegression` (interpretable baseline) vs
`RandomForestRegressor` (captures non-linear brand × capacity
interactions the linear model cannot).

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
    ),
}

results = {}

for model_name, model in models.items():

    pipeline = Pipeline(
        [("prep", preprocessor), ("model", model)]
    )

    cv_r2 = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="r2",
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results[model_name] = {
        "cv_r2_mean": cv_r2.mean(),
        "test_r2": r2_score(y_test, y_pred),
        "test_mae": mean_absolute_error(y_test, y_pred),
        "predictions": y_pred,
    }

    print(
        f"{model_name:18s} "
        f"CV R2 = {cv_r2.mean():.3f}  "
        f"Test R2 = {results[model_name]['test_r2']:.3f}  "
        f"MAE = {results[model_name]['test_mae']:,.0f} BDT"
    )

best_name = max(results, key=lambda k: results[k]["test_r2"])
print(f"\nBest model: {best_name}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

y_best = results[best_name]["predictions"]
axes[0].scatter(y_test, y_best, alpha=0.6, color="#4c72b0")
lims = [
    min(y_test.min(), y_best.min()),
    max(y_test.max(), y_best.max()),
]
axes[0].plot(lims, lims, "r--", linewidth=1)
axes[0].set_title(f"Actual vs predicted — {best_name}")
axes[0].set_xlabel("Actual price")
axes[0].set_ylabel("Predicted price")

fitted = Pipeline(
    [("prep", preprocessor), ("model", models[best_name])]
)
fitted.fit(X_train, y_train)
names = fitted.named_steps["prep"].get_feature_names_out()

if best_name == "RandomForest":
    importances = fitted.named_steps["model"].feature_importances_
    order = np.argsort(importances)[-10:]
    axes[1].barh(
        [names[i].split("__")[-1] for i in order],
        importances[order],
        color="#55a868",
    )
    axes[1].set_title("Top feature importances")
else:
    coefs = fitted.named_steps["model"].coef_
    order = np.argsort(np.abs(coefs))[-10:]
    axes[1].barh(
        [names[i].split("__")[-1] for i in order],
        coefs[order],
        color="#55a868",
    )
    axes[1].set_title("Top coefficients")

plt.tight_layout()
plt.show()

## 16 — Classification — Inverter vs Non-Inverter

Can we tell the AC type from brand, price, capacity and availability alone?
Evaluated with accuracy, per-class precision/recall and a confusion matrix.

In [ ]:
clf_features = ["brand", "availability", "price", "capacity_tons"]
Xc = df[clf_features]
yc = df["is_inverter"].astype(int)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc,
    yc,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=yc,
)

clf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            ["price", "capacity_tons"],
        ),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["brand", "availability"]),
    ]
)

classifiers = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
    ),
}

fitted_classifiers = {}

for clf_name, clf in classifiers.items():

    clf_pipeline = Pipeline(
        [("prep", clf_preprocessor), ("model", clf)]
    )

    clf_pipeline.fit(Xc_train, yc_train)
    yc_pred = clf_pipeline.predict(Xc_test)

    fitted_classifiers[clf_name] = clf_pipeline

    print(f"=== {clf_name} — accuracy: {accuracy_score(yc_test, yc_pred):.3f} ===")
    print(
        classification_report(
            yc_test,
            yc_pred,
            target_names=["Non-Inverter", "Inverter"],
            zero_division=0,
        )
    )

best_clf_name = max(
    fitted_classifiers,
    key=lambda n: accuracy_score(
        yc_test,
        fitted_classifiers[n].predict(Xc_test),
    ),
)

print(f"Best classifier: {best_clf_name}")

In [ ]:
yc_pred = fitted_classifiers[best_clf_name].predict(Xc_test)

cm = confusion_matrix(yc_test, yc_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], ["Non-Inverter", "Inverter"])
ax.set_yticks([0, 1], ["Non-Inverter", "Inverter"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion matrix — {best_clf_name}")

for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, str(value), ha="center", va="center")

plt.colorbar(im)
plt.tight_layout()
plt.show()

## 17 — Clustering (unsupervised) — market segments

K-Means on price + capacity + rating with the **elbow method**
for choosing k, then profile each cluster to interpret the segments.

In [ ]:
cluster_features = df[["price", "capacity_tons", "rating_average"]]
X_scaled = StandardScaler().fit_transform(cluster_features)

inertias = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(k_range), inertias, marker="o", color="#4c72b0")
plt.title("Elbow method — choosing k")
plt.xlabel("k (clusters)")
plt.ylabel("Inertia")
plt.tight_layout()
plt.show()

In [ ]:
K_BEST = 3

kmeans = KMeans(n_clusters=K_BEST, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

profile = (
    df.groupby("cluster")[
        ["price", "capacity_tons", "rating_average", "is_inverter"]
    ]
    .mean()
    .round(2)
)
profile["count"] = df["cluster"].value_counts().sort_index()
print("Cluster profiles (means):")
print(profile)

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    df["capacity_tons"],
    df["price"],
    c=df["cluster"],
    cmap="viridis",
    alpha=0.75,
)
ax.set_xlabel("Capacity (tons)")
ax.set_ylabel("Price (BDT)")
ax.set_title(f"K-Means segments (k={K_BEST})")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

## 18 — Course learning outcomes demonstrated

| Stage | What was done | Techniques / concepts |
|---|---|---|
| Data collection | Scraped live e-commerce catalog, stored via REST API (Part 1) | async crawling, rate limiting, selectors |
| Data flattening | Nested Product JSON → tidy table | feature extraction from raw schema |
| Data cleaning | Dedup, missing values, outlier capping, label canonicalization | `drop_duplicates`, IQR winsorization, regex recovery, canonical mapping |
| EDA | Distributions, category frequencies, boxplots | descriptive stats + matplotlib |
| Feature engineering | capacity from name/BTU, inverter & WiFi flags | domain-driven features |
| Preprocessing | impute + scale + one-hot in leak-free pipelines | `ColumnTransformer`, `Pipeline` |
| Regression | price prediction, model comparison | Linear vs Random Forest, 5-fold CV, MAE / R² |
| Classification | inverter detection | accuracy, precision / recall, confusion matrix |
| Clustering | market segmentation | K-Means, elbow method, cluster profiling |

**Data-mining takeaway:** the scraper + backend (Part 1) produce the
dataset; everything here — from the cleaning report to the cluster
profiles — is the data-mining workflow that turns that raw catalog
into insight (price drivers, product segments, class separation).